In [0]:
ECOMM_BRONZE_FILES = "abfss://bronze@gopaldatalake.dfs.core.windows.net/ecomm-gopal/"


dynamic partition prunning

Used when sub-quries return partition values, rather partition mentioned statically in the query

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Step 1: Define explicit schema
ecomm_schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True), # the csv does not have sql datetime format
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Country", StringType(), True)
])

# LAZY , not yet started the stream
# Step 1: Read from Bronze Zone
bronze_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(ecomm_schema)
    .load(ECOMM_BRONZE_FILES)  
)

# .show will not work on stream
bronze_df.printSchema()

bronze_df.select("Country").distinct().show(100)


In [0]:
from pyspark.sql import SparkSession

region_data = [
("Sweden","Europe"),
("Singapore","APAC"),
("Germany","Europe"),
("France","Europe"),
("Greece","Europe"),
("Belgium","Europe"),
("Finland","Europe"),
("Italy","Europe"),
("EIRE","Europe"),
("Lithuania","Europe"),
("Norway","Europe"),
("Spain","Europe"),
("Denmark","Europe"),
("Hong Kong","APAC"),
("Iceland","Europe"),
("Israel","Middle East"),
("Channel Islands","Europe"),
("Cyprus","Europe"),
("Saudi Arabia","Middle East"),
("Switzerland","Europe"),
("United Arab Emirates","Middle East"),
("Canada","North America"),
("Czech Republic","Europe"),
("Lebanon","Middle East"),
("Japan","APAC"),
("Poland","Europe"),
("Portugal","Europe"),
("Australia","APAC"),
("Austria","Europe"),
("Bahrain","Middle East"),
("United Kingdom","Europe"),
("Netherlands","Europe"),
("European Community","Europe"),
("Malta","Europe"),
("Unspecified","Other"),
("USA","North America"),
("Brazil","South America"),
("RSA","Africa")
]

regions_df = spark.createDataFrame(
    region_data,
    ["Country","Region"]
)

regions_df.show(truncate=False)

regions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("orderdb.regions")

In [0]:
%sql

SELECT * 
FROM orderdb.regions
ORDER BY Region, Country

In [0]:
%sql 
-- _ds data skipping demo
DROP TABLE IF EXISTS orderdb.gopal_ecomm_ds

In [0]:
(
bronze_df
.repartition(8)
.write
.format("delta")
.mode("overwrite")
.saveAsTable("orderdb.gopal_ecomm_ds")
)

In [0]:
%sql
-- NOT DPP, just a join query, it generally smaller table less than 10 mb broadcasted automatically
EXPLAIN FORMATTED
SELECT
    s.StockCode,
    SUM(s.Quantity) qty
FROM orderdb.gopal_ecomm_ds s
JOIN orderdb.regions r
ON s.Country = r.Country
WHERE r.Region='APAC'
GROUP BY s.StockCode

In [0]:
(
    bronze_df
      .repartition(20, "Country")
      .write
      .format("delta")
      .partitionBy("Country")
      .mode("overwrite")
      .saveAsTable("orderdb.gopal_ecomm_par")
)

In [0]:
%sql
--  DPP query, gopal_ecomm_par is partitioned
EXPLAIN FORMATTED
SELECT
    s.StockCode,
    SUM(s.Quantity) qty
FROM orderdb.gopal_ecomm_par s
JOIN orderdb.regions r
ON s.Country = r.Country
WHERE r.Region='APAC'
GROUP BY s.StockCode

-- check for PartitionFilters:  [isnotnull(Country#2281), dynamicpruning#2289 2287]
-- Join filter from regions produces country list
-- Spark injects dynamic pruning predicate
-- Only matching country partitions are scanned


In [0]:
%sql
ANALYZE TABLE  orderdb.gopal_ecomm_par COMPUTE DELTA STATISTICS

In [0]:
%sql
DESCRIBE DETAIL orderdb.gopal_ecomm_par